In [3]:
import pandas as pd
import pickle

from sympy import re
import re
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    
    # 1. Replace line breaks and tabs with a single space
    text = re.sub(r'[\r\n\t]+', ' ', text)
    
    # 2. Remove IP addresses (common in Wikipedia edits)
    text = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '', text)
    
    # 3. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 4. Standardize contraction apostrophes
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'s", " is", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'d", " would", text)
    
    # 5. Keep only alphanumeric characters and basic punctuation
    text = re.sub(r'[^a-zA-Z0-9\s!\?]', '', text)
    
    # 6. Collapse multiple whitespaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# --- 1. LOAD TEST IDs HASH SET ---
with open('valid_test_ids.pkl', 'rb') as f:
    valid_test_ids = pickle.load(f)

# --- 2. LOAD & CLEAN DATASETS WITH ORIGINAL IDs ---
# Load raw training data (retains 'id' column)
df_train = pd.read_csv('train.csv')
df_train['clean_text'] = df_train['comment_text'].apply(clean_text)

# Load test data and filter valid annotated rows using saved hash set
df_test = pd.read_csv('test.csv')
df_test = df_test[df_test['id'].isin(valid_test_ids)].copy()
df_test['clean_text'] = df_test['comment_text'].apply(clean_text)

# --- 3. INDEXING & OVERLAP CHECK VIA HASH LOOKUP ---
# Convert valid test texts into a lookup set for O(1) text matching
test_text_set = set(df_test['clean_text'])

# Check ID overlap
id_overlap = set(df_train['id']).intersection(valid_test_ids)

# Check Text overlap using boolean indexing on df_train
text_overlap_mask = df_train['clean_text'].isin(test_text_set)
common_text_df = df_train[text_overlap_mask]

# --- 4. DISPLAY RESULTS ---
print("==================================================")
print("       DATA LEAKAGE & OVERLAP CHECK SUMMARY       ")
print("==================================================")
print(f"Total Evaluated Test Samples : {len(df_test):,}")
print(f"Total Raw Training Samples   : {len(df_train):,}")
print("-" * 50)
print(f"1. Direct ID Overlap Count   : {len(id_overlap):,}")
print(f"2. Direct Text Overlap Count : {len(common_text_df):,}")
print("==================================================")

# --- 5. AUTOMATIC CLEANING IF LEAKAGE IS FOUND ---
if len(common_text_df) > 0:
    print(f"\n[!] Removing {len(common_text_df):,} overlapping texts from training set...")
    df_train_clean = df_train[~text_overlap_mask].reset_index(drop=True)
    print(f"Cleaned training shape: {df_train_clean.shape}")
else:
    print("\n[✓] PERFECT: Zero overlap detected between training and valid testing data!")

       DATA LEAKAGE & OVERLAP CHECK SUMMARY       
Total Evaluated Test Samples : 63,978
Total Raw Training Samples   : 159,571
--------------------------------------------------
1. Direct ID Overlap Count   : 0
2. Direct Text Overlap Count : 484

[!] Removing 484 overlapping texts from training set...
Cleaned training shape: (159087, 9)
